In [1]:
from platform import python_version
print(python_version())

3.11.14


### Open Targets

- disease 
- drug_mechanism_of_action 
- evidence_cancer_gene_census 
- evidence_reactome 
- interactions 
- pharmacogenomics 
- study

https://www.opentargets.org/

find a gene:
https://platform.opentargets.org/target/ENSG00000153395/associations

download:
https://platform.opentargets.org/downloads

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)


from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.config_lib import Config
from libs.open_target_lib import OpenTarget

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root_ot { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
               root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
               case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
               std_filename=std_filename, std_filename_list=std_filename_list,
               geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
               tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
               LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
               num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
               min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
               saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor
>>> case Tumor
	DEGs 20006
		Up (#10358)
		Dw (#9648)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    12
1                      IG_C_pseudogene     4
2                            IG_D_gene     1
3                            IG_J_gene     9
4                      IG_J_pseudogene     1
5                            IG_V_gene   124
6                      IG_V_pseudogene    50
7                              Mt_tRNA    17
8                                  TEC   112
9                            TR_C_gene     5
10                           TR_D_gene     1
11                           TR_J_gene    10
12                           TR_V_gene    69
13                     TR_V_pseudogene     1
14                              lncRNA  3193
15                               miRNA   

In [5]:
enr.root_kegg, enr.kegg_fname

(PosixPath('/home/flavio/uv/perturb_agent/data/colab/kegg'),
 'kegg_pathways.tsv')

In [6]:
enr.enr_db_list

['Reactome_Pathways_2024']

In [7]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

27223


,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


### Keneddy Pathway

https://www.wikipathways.org/pathways/WP3933.html

![Kennedy Pathway](../figures/WP3933_kennety_pathway.svg)

### Open Targets

```Bash
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_cancer_gene_census .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/drug_mechanism_of_action .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/known_drug .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/pharmacogenomics .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/study .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_reactome .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/association_by_datasource_direct .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/association_overall_direct .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/target .
colab/
    open_targets/
    ├── target/
    ├── disease/
    ├── drug_mechanism_of_action/
    ├── known_drug
    ├── evidence_cancer_gene_census/
    ├── evidence_reactome/
    ├── interactions/
    ├── pharmacogenomics/
    ├── association_by_datasource_direct/
    ├── association_overall_direct/
    └── study/
```

In [8]:
ot = OpenTarget(root_colab = root_colab)

### Is target ok?

In [9]:
ot.is_installed()

root exists: True
root is dir: True

Top-level content:
DIR  association_by_datasource_direct
DIR  association_overall_direct
DIR  disease
DIR  drug_mechanism_of_action
DIR  evidence_cancer_gene_census
DIR  evidence_reactome
FILE index.html
DIR  interactions
DIR  known_drug
DIR  pharmacogenomics
FILE robots.txt
DIR  study
DIR  target


In [10]:
ot.resolve_target("TP53")

,target_id,symbol,name,biotype
0,ENSG00000141510,TP53,tumor protein p53,protein_coding


### Testing

In [15]:
symbolA = "FOS"
symbolB = "JUN"
symbolC = None

target_idA, target_idB, df = ot.has_target_interactions(gene_or_ensemblA=symbolA, gene_or_ensemblB=symbolB)

print(len(df), f' interactions between {symbolA} ({target_idA}) and {symbolB} ({target_idB}).' )
df

5  interactions between FOS (ENSG00000170345) and JUN (ENSG00000177606).


,sourceDatabase,targetA,intA,intABiologicalRole,targetB,intB,intBBiologicalRole,speciesA,speciesB,count,scoring
0,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,competitor,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",1,0.980
1,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",44,0.980
2,reactome,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",3,NaN
3,signor,ENSG00000170345,P01100,regulator,ENSG00000177606,P05412,regulator target,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",1,NaN
4,string,ENSG00000170345,ENSP00000306245,unspecified role,ENSG00000177606,ENSP00000360266,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",4,0.999


In [16]:
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'CD274'
symbol = 'PTEN'
symbol = 'KLF5'
symbol = 'KRAS'


In [17]:
target_id, df = ot.has_target_moa(gene_or_ensembl=symbol)

print(len(df), f'moa found for target {symbol} {target_id}.' )
df.head(3)

3 moa found for target KRAS ENSG00000133703.


,chembl_id,mechanismOfAction,actionType,targetName,targetType,targets,references
0,CHEMBL23293,RAS inhibitor,INHIBITOR,RAS,protein family,"[ENSG00000174775, ENSG00000213281, ENSG00000133703]","[{'source': 'PubMed', 'ids': ['22547163'], 'urls': ['https://pubmed.ncbi.nlm..."
1,CHEMBL4535757,GTPase KRas inhibitor,INHIBITOR,GTPase KRas,single protein,[ENSG00000133703],"[{'source': 'PubMed', 'ids': ['31189530', '31666701', '31820981', '32568546'..."
2,CHEMBL4594350,GTPase KRas inhibitor,INHIBITOR,GTPase KRas,single protein,[ENSG00000133703],"[{'source': 'PubMed', 'ids': ['31658955'], 'urls': ['https://pubmed.ncbi.nlm..."


In [18]:
target_id, df = ot.get_reactome_pathways_for_target(symbol)

print(len(df), f'reactome evidences found for target {symbol} {target_id}.' )
df

71 reactome evidences found for target KRAS ENSG00000133703.


,targetId,approvedSymbol,approvedName,pathway_id,pathway,top_level_term
0,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-375165,NCAM signaling for neurite out-growth,Developmental Biology
1,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-5637810,Constitutive Signaling by EGFRvIII,Disease
2,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-1236382,Constitutive Signaling by Ligand-Responsive EGFR Cancer Variants,Disease
3,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-9634285,Constitutive Signaling by Overexpressed ERBB2,Disease
4,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-6802955,Paradoxical activation of RAF signaling by kinase inactive BRAF,Disease
...,...,...,...,...,...,...
66,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-112412,SOS-mediated signalling,Signal Transduction
67,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-1433557,Signaling by SCF-KIT,Signal Transduction
68,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-167044,Signalling to RAS,Signal Transduction
69,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-5218921,VEGFR2 mediated cell proliferation,Signal Transduction


In [38]:
target_id, df = ot.get_reactome_disease_evidence_for_target(symbol)

print(len(df), f'reactome disease evidences found for target {symbol} {target_id}.' )
df.head(3)


4 reactome disease evidences found for target TP53 ENSG00000141510.


,targetId,approvedSymbol,approvedName,diseaseId,disease_name,diseaseFromSource,pathway_id,pathway_name,reactionId,reactionName,targetModulation,score,literature,publicationDate,evidenceDate
0,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9723905,Loss of function of TP53 in cancer due to loss of tetramerization ability,R-HSA-9723906,Defective TP53 does not tetramerize,loss_of_function,1.0,"[7813439, 20978130, 9704930, 11753428, 9125151, 18940924, 10653977, 19913028...",1994-12-01,1994-12-01
1,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9723905,Loss of function of TP53 in cancer due to loss of tetramerization ability,R-HSA-9723906,Defective TP53 does not tetramerize,partial_loss_of_function,1.0,"[7813439, 20978130, 9704930, 11753428, 9125151, 18940924, 10653977, 19913028...",1994-12-01,1994-12-01
2,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9725370,Signaling by ALK fusions and activated point mutants,R-HSA-9851077,JNK-dependent inhibition of TP53 ubiquitination,up_or_down,1.0,[19286999],2009-03-13,2009-03-13


In [20]:
df = ot.con.execute(
    """
    SELECT
        t.id AS targetId,
        t.approvedSymbol,
        t.approvedName,

        p.pathwayId AS pathway_id,
        p.pathway AS pathway,
        p.topLevelTerm AS top_level_term

    FROM target t
    LEFT JOIN UNNEST(t.pathways) AS x(p) ON TRUE

    WHERE t.id = ?

    ORDER BY
        p.topLevelTerm NULLS LAST,
        p.pathway NULLS LAST
    """,
    [target_id],
).df()

df

,targetId,approvedSymbol,approvedName,pathway_id,pathway,top_level_term
0,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-375165,NCAM signaling for neurite out-growth,Developmental Biology
1,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-5637810,Constitutive Signaling by EGFRvIII,Disease
2,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-1236382,Constitutive Signaling by Ligand-Responsive EGFR Cancer Variants,Disease
3,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-9634285,Constitutive Signaling by Overexpressed ERBB2,Disease
4,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-6802955,Paradoxical activation of RAF signaling by kinase inactive BRAF,Disease
...,...,...,...,...,...,...
66,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-112412,SOS-mediated signalling,Signal Transduction
67,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-1433557,Signaling by SCF-KIT,Signal Transduction
68,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-167044,Signalling to RAS,Signal Transduction
69,ENSG00000133703,KRAS,"KRAS proto-oncogene, GTPase",R-HSA-5218921,VEGFR2 mediated cell proliferation,Signal Transduction


### Find a gene

In [21]:
print(ot.con.execute("SHOW TABLES").df())

                                name
0   association_by_datasource_direct
1         association_overall_direct
2                                cgc
3                            disease
4                           drug_moa
5                       interactions
6                         known_drug
7                   pharmacogenomics
8                           reactome
9                              study
10                            target


In [22]:
for table in ot.tables.keys():
    print("\n###", table, end=" ")
    print(ot.con.execute(f"SELECT COUNT(*) AS n FROM {table}").df())


### target        n
0  78766

### disease        n
0  47030

### drug_moa       n
0  6505

### known_drug         n
0  253459

### cgc        n
0  91572

### reactome        n
0  10748

### interactions           n
0  14618053

### pharmacogenomics        n
0  33080

### study          n
0  2001827

### association_by_datasource_direct          n
0  3826181

### association_overall_direct          n
0  3628026


In [23]:
ot.show_schema_summary()


### target


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,approvedSymbol,VARCHAR,YES,None,None,None
2,biotype,VARCHAR,YES,None,None,None
3,transcriptIds,VARCHAR[],YES,None,None,None
4,canonicalTranscript,"STRUCT(id VARCHAR, chromosome VARCHAR, ""start"" BIGINT, ""end"" BIGINT, strand ...",YES,None,None,None
5,canonicalExons,VARCHAR[],YES,None,None,None
6,genomicLocation,"STRUCT(chromosome VARCHAR, ""start"" BIGINT, ""end"" BIGINT, strand INTEGER)",YES,None,None,None
7,alternativeGenes,VARCHAR[],YES,None,None,None
8,approvedName,VARCHAR,YES,None,None,None
9,go,"STRUCT(id VARCHAR, ""source"" VARCHAR, evidence VARCHAR, aspect VARCHAR, geneP...",YES,None,None,None



### disease


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,name,VARCHAR,YES,None,None,None
3,description,VARCHAR,YES,None,None,None
4,dbXRefs,VARCHAR[],YES,None,None,None
5,parents,VARCHAR[],YES,None,None,None
6,exactSynonyms,VARCHAR[],YES,None,None,None
7,relatedSynonyms,VARCHAR[],YES,None,None,None
8,narrowSynonyms,VARCHAR[],YES,None,None,None
9,broadSynonyms,VARCHAR[],YES,None,None,None



### drug_moa


...

,column_name,column_type,null,key,default,extra
0,studyId,VARCHAR,YES,None,None,None
1,geneId,VARCHAR,YES,None,None,None
2,projectId,VARCHAR,YES,None,None,None
3,studyType,VARCHAR,YES,None,None,None
4,traitFromSource,VARCHAR,YES,None,None,None
5,traitFromSourceMappedIds,VARCHAR[],YES,None,None,None
6,biosampleFromSourceId,VARCHAR,YES,None,None,None
7,pubmedId,VARCHAR,YES,None,None,None
8,publicationTitle,VARCHAR,YES,None,None,None
9,publicationFirstAuthor,VARCHAR,YES,None,None,None



### association_by_datasource_direct


,column_name,column_type,null,key,default,extra
0,datatypeId,VARCHAR,YES,None,None,None
1,datasourceId,VARCHAR,YES,None,None,None
2,diseaseId,VARCHAR,YES,None,None,None
3,targetId,VARCHAR,YES,None,None,None
4,score,DOUBLE,YES,None,None,None
5,evidenceCount,BIGINT,YES,None,None,None



### association_overall_direct


,column_name,column_type,null,key,default,extra
0,diseaseId,VARCHAR,YES,None,None,None
1,targetId,VARCHAR,YES,None,None,None
2,score,DOUBLE,YES,None,None,None
3,evidenceCount,BIGINT,YES,None,None,None


### Then test target-specific retrieval with TP53:

In [24]:
symbol = 'TP53'
symbol = 'PCYT2'
symbol = 'CCNB2'
symbol = 'BLM'

# Em adenocarcinoma pancreático, o gene KRAS mutado ativa uma proteína chamada KLF5, que coordena simultaneamente a desregulação de quatro enzimas da via de síntese de membranas celulares — 
# criando um perfil metabólico identificável por biópsia com implicações prognósticas e terapêuticas.
symbol = 'KLF5'
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'KRAS'
symbol = 'CD274'
symbol = 'PTEN'

dft = ot.resolve_target(symbol)

if dft.empty:
    print("Nothing was found")
    target_id = ''
else:
    target_id = dft.iloc[0]["target_id"]

target_id

'ENSG00000171862'

### Test Cancer Gene Census

In [25]:
df = ot.is_target_related_to_cancer(target_id)

print(">>> target_id", target_id)
print(len(df), 'diseases found.')
df.head(2).T

>>> target_id ENSG00000171862
50 diseases found.


,0,1
targetId,ENSG00000171862,ENSG00000171862
id,bc28a1252ce28742e5edc7d772ed4e635ca1616e,ae908d5bb09180e9ca4010c9fb85b21020ceebbd
targetFromSourceId,ENSG00000171862,ENSG00000171862
diseaseFromSourceMappedId,EFO_0000209,EFO_1000098
datasourceId,cancer_gene_census,cancer_gene_census
datatypeId,somatic_mutation,somatic_mutation
diseaseFromSourceId,None,None
literature,"[25903014, 27655895, 24166518, 23349303, 33020650, 34362951, 28671688, 23341...","[9699651, 33016334, 10955808, 35953603, 10746673]"
mutatedSamples,"[{'functionalConsequenceId': 'SO_0001583', 'numberMutatedSamples': 299, 'num...","[{'functionalConsequenceId': 'SO_0001583', 'numberMutatedSamples': 28, 'numb..."
resourceScore,1.0,1.0


In [26]:
df = ot.is_target_related_to_disease(target_id=target_id, disease='melanoma')

print(len(df), 'diseases found.')
df.head(3)

38 diseases found.


,targetId,approvedSymbol,approvedName,diseaseId,disease_name,score,synonyms,ontology
0,ENSG00000171862,PTEN,phosphatase and tensin homolog,EFO_0000756,melanoma,0.754,"{'hasExactSynonym': ['Naevocarcinoma', 'malignant melanoma', 'melanoma', 'me...","{'isTherapeuticArea': False, 'leaf': False, 'sources': {'url': 'http://www.e..."
1,ENSG00000171862,PTEN,phosphatase and tensin homolog,EFO_0000389,cutaneous melanoma,0.688,"{'hasExactSynonym': ['cutaneous (skin) melanoma', 'cutaneous melanoma', 'cut...","{'isTherapeuticArea': False, 'leaf': False, 'sources': {'url': 'http://www.e..."
2,ENSG00000171862,PTEN,phosphatase and tensin homolog,MONDO_0008170,ovarian cancer,0.617,"{'hasExactSynonym': ['cancer of ovary', 'cancer of the ovary', 'malignant ne...","{'isTherapeuticArea': False, 'leaf': False, 'sources': {'url': 'http://purl...."


In [27]:
ot.schema("drug_moa")

,column_name,column_type,null,key,default,extra
0,actionType,VARCHAR,YES,None,None,None
1,mechanismOfAction,VARCHAR,YES,None,None,None
2,chemblIds,VARCHAR[],YES,None,None,None
3,targetName,VARCHAR,YES,None,None,None
4,targetType,VARCHAR,YES,None,None,None
5,targets,VARCHAR[],YES,None,None,None
6,references,"STRUCT(""source"" VARCHAR, ids VARCHAR[], urls VARCHAR[])[]",YES,None,None,None


In [28]:
df = ot.con.execute(
    """
    SELECT
        kd.drugId,
        kd.prefName,
        kd.targetId,
        moa.chemblIds,
        moa.targets,
        moa.mechanismOfAction,
        moa.actionType
    FROM known_drug kd
    LEFT JOIN drug_moa moa
        ON list_contains(moa.chemblIds, kd.drugId)
       AND list_contains(moa.targets, kd.targetId)
    WHERE kd.drugId IS NOT NULL
    LIMIT 20
    """
).df()

df.head(3)


,drugId,prefName,targetId,chemblIds,targets,mechanismOfAction,actionType
0,CHEMBL862,GUANFACINE,ENSG00000184160,"[CHEMBL1200494, CHEMBL862]","[ENSG00000150594, ENSG00000184160, ENSG00000274286]",Adrenergic receptor alpha-2 agonist,AGONIST
1,CHEMBL862,GUANFACINE,ENSG00000150594,"[CHEMBL1200494, CHEMBL862]","[ENSG00000150594, ENSG00000184160, ENSG00000274286]",Adrenergic receptor alpha-2 agonist,AGONIST
2,CHEMBL134,CLONIDINE,ENSG00000274286,[CHEMBL134],"[ENSG00000150594, ENSG00000184160, ENSG00000274286]",Adrenergic receptor alpha-2 agonist,AGONIST


In [29]:
disease='melanoma'

df = ot.get_drugs_for_disease(disease=disease, limit=None)

print(len(df), f'drugs found for {disease}.')
df.head(2).T


9477 drugs found for melanoma.


,0,1
diseaseId,EFO_0002617,EFO_0002617
disease_name,metastatic melanoma,metastatic melanoma
drugId,CHEMBL1201438,CHEMBL1201438
drugName,ALDESLEUKIN,ALDESLEUKIN
targetId,ENSG00000134460,ENSG00000134460
approvedSymbol,IL2RA,IL2RA
approvedName,interleukin 2 receptor subunit alpha,interleukin 2 receptor subunit alpha
phase,4.0,4.0
status,Completed,None
urls,"[{'niceName': 'ClinicalTrials', 'url': 'https://clinicaltrials.gov/study/NCT...","[{'niceName': 'DailyMed', 'url': 'https://dailymed.nlm.nih.gov/dailymed/drug..."


In [30]:
df = ot.get_drugs_for_disease_with_moa(disease=disease, limit=None)

print(len(df), f'drugs found for {disease}.')
df.head(2).T

10814 drugs found for melanoma.


,0,1
diseaseId,EFO_0002617,EFO_0002617
disease_name,metastatic melanoma,metastatic melanoma
drugId,CHEMBL1201438,CHEMBL1201438
drugName,ALDESLEUKIN,ALDESLEUKIN
targetId,ENSG00000134460,ENSG00000134460
approvedSymbol,IL2RA,IL2RA
approvedName,interleukin 2 receptor subunit alpha,interleukin 2 receptor subunit alpha
phase,4.0,4.0
status,Terminated,Completed
urls,"[{'niceName': 'ClinicalTrials', 'url': 'https://clinicaltrials.gov/study/NCT...","[{'niceName': 'ClinicalTrials', 'url': 'https://clinicaltrials.gov/study/NCT..."


### Test Reactome evidence

In [31]:
ot.schema("reactome")

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,targetFromSourceId,VARCHAR,YES,None,None,None
2,diseaseFromSourceMappedId,VARCHAR,YES,None,None,None
3,datasourceId,VARCHAR,YES,None,None,None
4,datatypeId,VARCHAR,YES,None,None,None
5,diseaseFromSource,VARCHAR,YES,None,None,None
6,diseaseFromSourceId,VARCHAR,YES,None,None,None
7,literature,VARCHAR[],YES,None,None,None
8,pathways,"STRUCT(id VARCHAR, ""name"" VARCHAR)[]",YES,None,None,None
9,reactionId,VARCHAR,YES,None,None,None


In [32]:
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'KRAS'
symbol = 'CD274'
symbol = 'KLF5'
symbol = 'TP53'

dft = ot.resolve_target(symbol)

if dft.empty:
    print("Nothing was found")
    target_id = ''
else:
    target_id = dft.iloc[0]["target_id"]

print(">>>", target_id)

>>> ENSG00000141510


In [33]:
target_id, df = ot.get_reactome_pathways_for_target(gene_or_ensembl=symbol)

print(len(df), f'reactome evidences found for target {symbol} {target_id}.' )
df.head(3)

45 reactome evidences found for target TP53 ENSG00000141510.


,targetId,approvedSymbol,approvedName,pathway_id,pathway,top_level_term
0,ENSG00000141510,TP53,tumor protein p53,R-HSA-349425,Autodegradation of the E3 ubiquitin ligase COP1,Cell Cycle
1,ENSG00000141510,TP53,tumor protein p53,R-HSA-69481,G2/M Checkpoints,Cell Cycle
2,ENSG00000141510,TP53,tumor protein p53,R-HSA-69473,G2/M DNA damage checkpoint,Cell Cycle


In [34]:
target_id, df = ot.get_reactome_disease_evidence_for_target(gene_or_ensembl=symbol)

print(len(df), f'reactome disease evidences found for target {symbol} {target_id}.' )
df.head(3)

4 reactome disease evidences found for target TP53 ENSG00000141510.


,targetId,approvedSymbol,approvedName,diseaseId,disease_name,diseaseFromSource,pathway_id,pathway_name,reactionId,reactionName,targetModulation,score,literature,publicationDate,evidenceDate
0,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9723905,Loss of function of TP53 in cancer due to loss of tetramerization ability,R-HSA-9723906,Defective TP53 does not tetramerize,loss_of_function,1.0,"[7813439, 20978130, 9704930, 11753428, 9125151, 18940924, 10653977, 19913028...",1994-12-01,1994-12-01
1,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9723905,Loss of function of TP53 in cancer due to loss of tetramerization ability,R-HSA-9723906,Defective TP53 does not tetramerize,partial_loss_of_function,1.0,"[7813439, 20978130, 9704930, 11753428, 9125151, 18940924, 10653977, 19913028...",1994-12-01,1994-12-01
2,ENSG00000141510,TP53,tumor protein p53,MONDO_0004992,cancer,cancer,R-HSA-9725370,Signaling by ALK fusions and activated point mutants,R-HSA-9851077,JNK-dependent inhibition of TP53 ubiquitination,up_or_down,1.0,[19286999],2009-03-13,2009-03-13


### drug mechanism of action

In [35]:
target_id, df = ot.has_target_moa(gene_or_ensembl=symbol)

print(len(df), f'moa found for target {symbol} {target_id}.' )
df.head(3)

10 moa found for target TP53 ENSG00000141510.


,chembl_id,mechanismOfAction,actionType,targetName,targetType,targets,references
0,CHEMBL2108280,p53 mRNA antisense inhibitor,ANTISENSE INHIBITOR,p53 mRNA,nucleic-acid,[ENSG00000141510],"[{'source': 'PubMed', 'ids': ['21717444', '22944355'], 'urls': ['http://euro..."
1,CHEMBL2108585,Cellular tumor antigen p53 exogenous gene,EXOGENOUS GENE,Cellular tumor antigen p53,single protein,[ENSG00000141510],"[{'source': 'EMA', 'ids': ['https://www.ema.europa.eu/en/medicines/human/wit..."
2,CHEMBL2219776,p53 mRNA antisense inhibitor,ANTISENSE INHIBITOR,p53 mRNA,nucleic-acid,[ENSG00000141510],"[{'source': 'PubMed', 'ids': ['21717444', '22944355'], 'urls': ['http://euro..."


### interactions

In [36]:
ot.schema("interactions")

,column_name,column_type,null,key,default,extra
0,sourceDatabase,VARCHAR,YES,None,None,None
1,targetA,VARCHAR,YES,None,None,None
2,intA,VARCHAR,YES,None,None,None
3,intABiologicalRole,VARCHAR,YES,None,None,None
4,targetB,VARCHAR,YES,None,None,None
5,intB,VARCHAR,YES,None,None,None
6,intBBiologicalRole,VARCHAR,YES,None,None,None
7,speciesA,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
8,speciesB,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
9,count,BIGINT,YES,None,None,None


In [37]:
symbolA = "FOS"
symbolB = "JUN"

target_idA, target_idB, df = ot.has_target_interactions(gene_or_ensemblA=symbolA, gene_or_ensemblB=symbolB)

print(len(df), f' interactions between {symbolA} ({target_idA}) and {symbolB} ({target_idB}).' )
df.head(3)

5  interactions between FOS (ENSG00000170345) and JUN (ENSG00000177606).


,sourceDatabase,targetA,intA,intABiologicalRole,targetB,intB,intBBiologicalRole,speciesA,speciesB,count,scoring
0,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,competitor,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",1,0.98
1,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",44,0.98
2,reactome,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",3,NaN
